# Notebook 2: Data Preprocessing
## Project: Crop Stress and Irrigation Priority Decision Support System
**Objective:** Clean raw features, handle missing values, remove duplicates, select relevant predictors, encode target labels, and prepare training matrices without data leakage.

### Step 1: Import Required Libraries
We import Pandas and NumPy for tabular operations.

In [1]:
import pandas as pd
import numpy as np

### Step 2: Load the Master Dataset

In [2]:
try:
    df = pd.read_csv("data/processed/krishijal_master_dataset.csv")
    print("Loaded dataset from local path: data/processed/krishijal_master_dataset.csv")
except:
    try:
        df = pd.read_csv("../data/processed/krishijal_master_dataset.csv")
        print("Loaded dataset from parent path: ../data/processed/krishijal_master_dataset.csv")
    except:
        df = pd.read_csv("/kaggle/input/krishijal-dataset/krishijal_master_dataset.csv")
        print("Loaded dataset from Kaggle path: /kaggle/input/krishijal-dataset/krishijal_master_dataset.csv")

Loaded dataset from parent path: ../data/processed/krishijal_master_dataset.csv


### Step 3: Remove Duplicate Rows
We verify that no duplicate rows exist in our working DataFrame.

In [3]:
initial_rows = len(df)
df = df.drop_duplicates()
print("Rows before deduplication:", initial_rows)
print("Rows after deduplication:", len(df))
print("Removed duplicates:", initial_rows - len(df))

Rows before deduplication: 6654
Rows after deduplication: 6654
Removed duplicates: 0


### Step 4: Handle Missing Values
For continuous agro-meteorological features, any missing value is imputed using the median of that feature. For categorical columns, we use the mode.

In [4]:
numeric_features = [
    "temperature", "temp_max", "temp_min", "rainfall", "humidity",
    "effective_soil_wetness", "solar_radiation", "soil_ph", "soil_organic_carbon",
    "ndvi", "et0", "etc", "cwsi", "water_deficit", "irrigation_priority_index"
]

for col in numeric_features:
    df[col] = df[col].fillna(df[col].median())

print("Missing values in core numeric features after median imputation:")
print(df[numeric_features].isnull().sum())

Missing values in core numeric features after median imputation:
temperature                  0
temp_max                     0
temp_min                     0
rainfall                     0
humidity                     0
effective_soil_wetness       0
solar_radiation              0
soil_ph                      0
soil_organic_carbon          0
ndvi                         0
et0                          0
etc                          0
cwsi                         0
water_deficit                0
irrigation_priority_index    0
dtype: int64


### Step 5: Feature Selection
Farmers cannot easily measure 72 columns. To build a practical, reliable decision support system, we select the 9 most influential physical and meteorological predictors:
1. `cwsi`: Crop Water Stress Index (0.0 to 1.0)
2. `effective_soil_wetness`: Weighted root-zone and surface soil moisture (0.0 to 1.0)
3. `water_deficit`: Daily crop water deficit in mm/day
4. `ndvi`: Canopy greenness from satellite remote sensing (0.1 to 0.9)
5. `et0`: Reference evapotranspiration (mm/day)
6. `etc`: Crop evapotranspiration requirement (mm/day)
7. `temperature`: Mean air temperature (°C)
8. `rainfall`: Daily rainfall (mm)
9. `humidity`: Relative humidity (%)

In [5]:
selected_features = [
    "cwsi", "effective_soil_wetness", "water_deficit", "ndvi",
    "et0", "etc", "temperature", "rainfall", "humidity"
]

X = df[selected_features].copy()
print("Selected Feature Matrix Shape:", X.shape)
X.head()

Selected Feature Matrix Shape: (6654, 9)


,cwsi,effective_soil_wetness,water_deficit,ndvi,et0,etc,temperature,rainfall,humidity
0,0.088,0.502,0.08,0.703,3.25,3.41,23.94,0.22,69.06
1,0.234,0.422,0.68,0.655,4.19,4.40,26.68,0.35,61.54
2,0.320,0.374,1.34,0.626,4.88,5.12,30.04,0.30,56.89
3,0.319,0.374,1.15,0.626,5.43,5.70,32.36,0.67,56.21
4,0.225,0.426,0.00,0.658,5.25,5.51,33.69,1.55,54.54


### Step 6: Target Variables Extraction & Categorical Encoding
We prepare two target variables:
1. **Classification Target:** `stress_level` mapped to integers:
   - `0: Low Stress`
   - `1: Moderate Stress`
   - `2: High Stress`
   - `3: Severe Stress`
2. **Regression Target:** `irrigation_priority_index` (Continuous score from 0.0 to 100.0).

In [6]:
stress_mapping = {
    "Low": 0,
    "Moderate": 1,
    "High": 2,
    "Severe": 3
}

y_classification = df["stress_level"].map(stress_mapping)
y_regression = df["irrigation_priority_index"]

print("Classification Target Value Counts (Encoded):")
print(y_classification.value_counts())
print("\nRegression Target Summary:")
print(y_regression.describe())

Classification Target Value Counts (Encoded):
stress_level
0    4984
1     910
2     566
3     194
Name: count, dtype: int64

Regression Target Summary:
count    6654.000000
mean       21.699970
std        13.942208
min         0.800000
25%        10.000000
50%        19.000000
75%        28.875000
max        73.700000
Name: irrigation_priority_index, dtype: float64


### Step 7: Handle Outliers with Physical Bounding
In agriculture, physical indices have defined bounds:
- CWSI must lie between 0.0 and 1.0.
- Effective soil wetness must lie between 0.0 and 1.0.
- NDVI must lie between 0.10 and 0.90.
We clip features to their physically valid boundaries.

In [7]:
X["cwsi"] = X["cwsi"].clip(0.0, 1.0)
X["effective_soil_wetness"] = X["effective_soil_wetness"].clip(0.0, 1.0)
X["ndvi"] = X["ndvi"].clip(0.10, 0.90)
X["water_deficit"] = X["water_deficit"].clip(lower=0.0)

print("Physical boundaries applied successfully.")
X.describe()

Physical boundaries applied successfully.


,cwsi,effective_soil_wetness,water_deficit,ndvi,et0,etc,temperature,rainfall,humidity
count,6654.000000,6654.000000,6654.000000,6654.000000,6654.000000,6654.000000,6654.000000,6654.000000,6654.000000
mean,0.087087,0.630155,0.259997,0.672605,4.086335,4.024665,25.745818,4.557325,70.246515
std,0.140770,0.170067,0.613972,0.176317,1.145563,1.292759,5.535395,9.270983,16.791530
min,0.000000,0.130000,0.000000,0.186000,0.100000,0.090000,-16.010000,0.000000,16.610000
25%,0.000000,0.495000,0.000000,0.580000,3.340000,3.100000,23.582500,0.050000,60.872500
50%,0.000000,0.642000,0.000000,0.724000,4.130000,3.970000,26.570000,1.030000,75.280000
75%,0.156000,0.765000,0.000000,0.783000,4.790000,4.900000,28.840000,5.640000,83.217500
max,0.764000,0.990000,4.950000,0.880000,7.540000,8.180000,40.990000,262.610000,96.070000


### Step 8: Save Clean Preprocessed Dataset
We assemble the clean feature matrix and targets into `data/processed/krishijal_ml_clean.csv` for modeling.

In [8]:
clean_ml_df = X.copy()
clean_ml_df["stress_level_code"] = y_classification
clean_ml_df["irrigation_priority_index"] = y_regression

# Save to data/processed or local directory depending on execution path
try:
    clean_ml_df.to_csv("data/processed/krishijal_ml_clean.csv", index=False)
    print("Saved clean dataset to: data/processed/krishijal_ml_clean.csv")
except:
    try:
        clean_ml_df.to_csv("../data/processed/krishijal_ml_clean.csv", index=False)
        print("Saved clean dataset to: ../data/processed/krishijal_ml_clean.csv")
    except:
        clean_ml_df.to_csv("krishijal_ml_clean.csv", index=False)
        print("Saved clean dataset to: krishijal_ml_clean.csv")

print("Clean Dataset Shape:", clean_ml_df.shape)

Saved clean dataset to: ../data/processed/krishijal_ml_clean.csv
Clean Dataset Shape: (6654, 11)


### Step 9: Summary of Preprocessing
- Duplicates checked and confirmed clean.
- Missing values imputed using feature medians.
- 9 core agricultural predictors selected.
- Target encoded deterministically without data leakage.
- Clean dataset saved and ready for Exploratory Data Analysis (`03_eda.ipynb`).